# Modelling

## Import Libraries

In [1]:
# Import standard libraries
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from category_encoders import BinaryEncoder

## Load Final Dataset

In [2]:
DATA_PATH = '../data/processed/combined_final_dataset.csv'

df = pd.read_csv(DATA_PATH)
print(f"Loaded prepared dataset: {df.shape[0]} rows x {df.shape[1]} columns")
df.head()

Loaded prepared dataset: 680694 rows x 11 columns


,country,make,model,year,age,mileage,transmission,fuelType,mpg,engineSize,price
0,Uk,Vauxhall,Corsa,2018,2,9876.0,Manual,Petrol,46.13158,1.4,10124.340
1,Uk,Vauxhall,Corsa,2019,1,2500.0,Manual,Petrol,45.21561,1.4,15401.580
2,Uk,Vauxhall,Corsa,2017,3,9625.0,Automatic,Petrol,39.88633,1.4,12553.668
3,Uk,Vauxhall,Corsa,2016,4,25796.0,Manual,Petrol,46.13158,1.4,10914.000
4,Uk,Vauxhall,Corsa,2019,1,3887.0,Manual,Petrol,36.22245,1.4,12840.000


## Select Models

### Modelling Techniques

In [3]:
modelling_techniques = {
    "problem_type": "Regression",
    "target_variable": "price",
    "candidate_models": [
        {"name": "Linear Regression", "justification": "Simple Baseline Model Used to Reveal relationsips between features and price"},
        {"name": "Ridge Regression", "justification": "Manages related features better then linear regression such as engineSize and mpg"},
        {"name": "Lasso Regression", "justification": "Better at filtering features with the least impact such as country"},
        {"name": "Decision Tree", "justification": "Split data by features and views price differences between cars by features example being manual vs automatic car transmissions"},
        {"name": "Random Forest", "justification": "Stops outliner from ruining accuracy making the model produce more stable results"},
        {"name": "Gradient Boosting", "justification": "Learn how specific features influence price through sequential steps, Produce high accuracy for car valuations"},
        {"name": "KNN Regressor", "justification": "Find cars with identical features such as mileage and fuel types. Calculate prices by averaging neighbors."}
    ]
}

print(f"Problem Type: {modelling_techniques['problem_type']}")
print(f"Target Variable: {modelling_techniques['target_variable']}")
print(f"\nCandidate Models:")
for i, model in enumerate(modelling_techniques['candidate_models'], 1):
    print(f"  {i}. {model['name']} — {model['justification']}")

Problem Type: Regression
Target Variable: price

Candidate Models:
  1. Linear Regression — Simple Baseline Model Used to Reveal relationsips between features and price
  2. Ridge Regression — Manages related features better then linear regression such as engineSize and mpg
  3. Lasso Regression — Better at filtering features with the least impact such as country
  4. Decision Tree — Split data by features and views price differences between cars by features example being manual vs automatic car transmissions
  5. Random Forest — Stops outliner from ruining accuracy making the model produce more stable results
  6. Gradient Boosting — Learn how specific features influence price through sequential steps, Produce high accuracy for car valuations
  7. KNN Regressor — Find cars with identical features such as mileage and fuel types. Calculate prices by averaging neighbors.


## Data Preprocessing

### Handle year column, and EV mpg and engineSize

In [4]:
# Set random seed and split test ratio according to test
RANDOM_SEED = 42
TEST_SIZE = 0.2

# Drop rows where target is missing or invalid
df = df.dropna(subset=['price'])
df = df[df['price'] > 0]
print(f"Dataset after filtering: {df.shape}")

# Drop year as it is correlated with the age of the car made in the data preparation step
if 'year' in df.columns:
    df = df.drop(columns=['year'])
    print("Dropped 'year' column")

# Create is_ev flag to make models ignore mpg and engineSize
df['is_ev'] = (df['fuelType'] == 'Electric').astype(int)
print(f"EV records: {df['is_ev'].sum()} / {len(df)} ({df['is_ev'].mean()*100:.2f}%)")

# Replace mpg and engineSize with median of both from petrol cars
non_ev_mpg_median = df.loc[df['is_ev'] == 0, 'mpg'].median()
non_ev_engine_median = df.loc[df['is_ev'] == 0, 'engineSize'].median()

df.loc[df['is_ev'] == 1, 'mpg'] = non_ev_mpg_median
df.loc[df['is_ev'] == 1, 'engineSize'] = non_ev_engine_median
print(f"Imputed EV mpg with {non_ev_mpg_median:.1f}, engineSize with {non_ev_engine_median:.1f}")

Dataset after filtering: (680694, 11)
Dropped 'year' column
EV records: 2005 / 680694 (0.29%)
Imputed EV mpg with 25.5, engineSize with 2.4


### Set train/test split

In [5]:
# Define features and target
TARGET = 'price'
X = df.drop(columns=[TARGET])
y = df[TARGET]

# Initialize Train/test split with ev stratify to count ev in the split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_SEED,
    stratify=X['is_ev']
)
print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set:     {X_test.shape[0]} samples")

# Define sample weights to make UK more prominent
train_weights = np.where(X_train['country'] == 'UK', 4.0, 0.5)
print("\nSample weights computed (UK=4.0, US=0.5).")

Training set: 544555 samples
Test set:     136139 samples

Sample weights computed (UK=4.0, US=0.5).


### Encoding Categorical Features

#### One-Hot Encode Low Cardinality Features

In [6]:
# Apply One-Hot Encoding to prevent linear/KNN models from inferring false ordinality
low_cat_features = ['country', 'transmission', 'fuelType']
X_train = pd.get_dummies(X_train, columns=low_cat_features, drop_first=True)
X_test = pd.get_dummies(X_test, columns=low_cat_features, drop_first=True)

# Align columns to ensure test set has exactly the same dummy columns as the training set
X_train, X_test = X_train.align(X_test, join='lef   t', axis=1, fill_value=0)

#### Target Encode High Cardinality Features

In [7]:
high_cat_features = ['make', 'model']

# Initialize BinaryEncoder to condense high-cardinality features into bit-columns
binary_encoder = BinaryEncoder(cols=high_cat_features)

# Fit and transform the training data
X_train = binary_encoder.fit_transform(X_train)
X_test = binary_encoder.transform(X_test)

# Align columns to ensure test set has exactly the same binary columns as the training set
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

#### Scaling Numerical Features

In [8]:
scaler = StandardScaler()

# Keep output as DataFrames to retain column names for Feature Importance analysis later
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

print(f"\nFeature matrix shape: {X_train_scaled.shape}")


Feature matrix shape: (544555, 33)


### Test Design

In [9]:
# Document the test design
test_design = {
    "split_ratio": f"{int((1 - TEST_SIZE) * 100)}/{int(TEST_SIZE * 100)}",
    "validation_strategy": "5-fold",
    "random_seed": RANDOM_SEED,
    "encoding": "One-Hot for low-cardinality; Binary Encoding for high-cardinality (Make, Model)",
    "missing_value_strategy": "Median imputation; EV mpg/engineSize set to ICE medians",
    "justification": "80/20 split provides sufficient test data while maximising training data."
}

for key, value in test_design.items():
    print(f"{key}: {value}")

split_ratio: 80/20
validation_strategy: 5-fold
random_seed: 42
encoding: One-Hot for low-cardinality; Binary Encoding for high-cardinality (Make, Model)
missing_value_strategy: Median imputation; EV mpg/engineSize set to ICE medians
justification: 80/20 split provides sufficient test data while maximising training data.


### Model Build

In [10]:
# Initialize dictionary to store fitted model objects
trained_models = {}

# Initialize a map with the regression models that would be used
models_to_train = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0, random_state=RANDOM_SEED),
    'Lasso Regression': Lasso(alpha=1.0, random_state=RANDOM_SEED),
    'Decision Tree': DecisionTreeRegressor(random_state=RANDOM_SEED, max_depth=15),
    'Random Forest': RandomForestRegressor(random_state=RANDOM_SEED, n_estimators=100, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(random_state=RANDOM_SEED, n_estimators=100),
    'KNN Regressor': KNeighborsRegressor(n_neighbors=5, n_jobs=-1)
}

# Train each model
for name, model in models_to_train.items():
    print(f"Training {name}...")
    
    # Check for KNN to not use sample_weight
    if name == 'KNN Regressor':
        model.fit(X_train_scaled, y_train)
    else:
        model.fit(X_train_scaled, y_train, sample_weight=train_weights)

    # Store model in a map
    trained_models[name] = model
    print(f"{name} trained successfully.")

print(f"\nSuccessfully trained {len(trained_models)} model(s).")

Training Linear Regression...
Linear Regression trained successfully.
Training Ridge Regression...
Ridge Regression trained successfully.
Training Lasso Regression...
Lasso Regression trained successfully.
Training Decision Tree...
Decision Tree trained successfully.
Training Random Forest...
Random Forest trained successfully.
Training Gradient Boosting...
Gradient Boosting trained successfully.
Training KNN Regressor...
KNN Regressor trained successfully.

Successfully trained 7 model(s).


### Model Assessment

#### R2, MAE and RMSE

In [11]:
# Initialize results list to capture performance data for each model
results = []

# Iterate through each fitted model to evaluate its predictive power on the unseen test set
for name, model in trained_models.items():
    # Generate predictions using the scaled features to maintain consistency with training
    y_pred = model.predict(X_test_scaled)

    # Calculate standard regression metrics: MAE (error magnitude), RMSE (penalizes outliers), and R2 (variance explained)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    
    # Append metrics to the results list for tabular comparison
    results.append({
        'Model': name,
        'MAE': mae,
        'RMSE': rmse,
        'R2 Score': r2
    })

# Convert the results list into a DataFrame and sort by MAE to highlight the best-performing model
results_df = pd.DataFrame(results).set_index('Model').sort_values(by='MAE')

# Display the final comparison table
print("=== Model Comparison (Sorted by Lowest MAE) ===")
print(results_df.round(4))

=== Model Comparison (Sorted by Lowest MAE) ===
                         MAE       RMSE  R2 Score
Model                                            
KNN Regressor      1942.1593  2931.2117    0.9436
Random Forest      1991.7812  2898.9568    0.9448
Decision Tree      2448.7172  3647.5672    0.9127
Gradient Boosting  4028.6337  5610.9700    0.7934
Ridge Regression   5174.5175  7228.3877    0.6571
Linear Regression  5174.5185  7228.3877    0.6571
Lasso Regression   5174.5212  7228.3957    0.6571


#### 5-Fold CV

In [12]:
# Grab Best Model from Results Dataframe
best_model_name = results_df.index[0]
best_model = trained_models[best_model_name]

# Execute cross-validation on the top-performing model to ensure the performance is consistent
cv_scores = cross_val_score(
    best_model, 
    X_train_scaled, 
    y_train, 
    cv=5,
    scoring='neg_mean_absolute_error', 
    n_jobs=-1
)

# Convert negative MAE back to positive values for easier interpretation
cv_mae_scores = -cv_scores

# Print the individual fold results and the aggregate statistics
print(f"Cross-Validation MAE Scores across folds: {cv_mae_scores.round(2)}")
print(f"Mean CV MAE: {cv_mae_scores.mean():.2f} (+/- {cv_mae_scores.std():.2f} standard deviation)")

Cross-Validation MAE Scores across folds: [1976.68 1971.14 1978.41 1974.78 1965.78]
Mean CV MAE: 1973.36 (+/- 4.49 standard deviation)


#### Feature Importance (Best Model)

In [15]:
# Initialize feature importance and feature names
importances = best_model.feature_importances_
feature_names = X_train_scaled.columns

# Wrap the feature names and importances in a dataframe
feature_importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

# Show the features and importance values in the console
print(feature_importance_df.head(10))

AttributeError: 'KNeighborsRegressor' object has no attribute 'feature_importances_'